# Notebook de Simulação e Teste para o LiveTrader (Multi-Ativo)

Este notebook permite executar o fluxo de trabalho do `LiveTrader` passo a passo para um **único ativo escolhido**, ideal para depuração e simulações, incluindo a verificação do ticker de ordem para contratos futuros.

**Pré-requisitos:**
1. O terminal MetaTrader 5 deve estar aberto e logado.
2. Os modelos de produção devem ter sido gerados pelo script `train_model.py`.

In [13]:
import pandas as pd
import yaml
import sys
from pathlib import Path
import MetaTrader5 as mt5
import time
from datetime import datetime, timezone
import logging

logging.basicConfig(level=logging.INFO)

# Adiciona a pasta 'src' ao path para permitir as importações dos nossos módulos
project_root = Path.cwd().parent.parent  # going up one more level to reach the root
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.live_trader import LiveTrader

### Passo 1: Escolha o Ativo e Inicialize o Trader

**Ação:** Defina a variável `DATA_TICKER_PARA_TESTAR` com o ticker de DADOS HISTÓRICOS (ex: "WDO$").

In [14]:
# --- ESCOLHA O ATIVO AQUI (use o ticker de dados históricos) ---
DATA_TICKER_PARA_TESTAR = "WDO$"
# ------------------------------------------------------------

trader = LiveTrader(config_path='configs/main.yaml')
is_initialized = trader.initialize()

if is_initialized and DATA_TICKER_PARA_TESTAR in trader.asset_states:
    print(f"Trader inicializado com sucesso. Foco do teste: {DATA_TICKER_PARA_TESTAR}")
    asset_state_para_testar = trader.asset_states[DATA_TICKER_PARA_TESTAR]
else:
    print(f"Falha ao inicializar ou ticker '{DATA_TICKER_PARA_TESTAR}' não encontrado/carregado.")
    asset_state_para_testar = None

2025-10-10 22:58:06,188 - INFO - Diretório de cache de dados inicializado em: C:\projects\wtnps-trade\notebooks\simulation\.cache_data
2025-10-10 22:58:06,193 - INFO - Inicializando o robô trader multi-ativo...
2025-10-10 22:58:06,199 - INFO -   > Carregando recursos para WDO$...
2025-10-10 22:58:06,201 - INFO - Carregando modelo de c:\projects\wtnps-trade\models\WDO$_prod_model.keras e scaler de c:\projects\wtnps-trade\models\WDO$_prod_scaler.joblib
c:\Users\User\AppData\Local\pypoetry\Cache\virtualenvs\wtnps-trade-VyqtAXyS-py3.12\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
2025-10-10 22:58:06,552 - INFO -   > Carregando recursos para WIN$...
2025-10-10 22:58:06,554 - INFO - Carregando modelo de c:\projects\wtnps-trade\models\WIN$_prod_model.keras e scaler de c:\projec

Trader inicializado com sucesso. Foco do teste: WDO$


### Passo 2: Método para executar um Único Ciclo de Decisão (Single Tick)

In [18]:
def run_single_tick(trader_instance, asset_state):
    if not asset_state:
        print("Estado do ativo não inicializado.")
        return

    data_ticker = asset_state['config']['ticker']
    order_ticker = asset_state['config']['live_trading']['ticker_order']
    timeframe_str = asset_state['config']['live_trading']['timeframe_str']
    mt5_timeframe = trader_instance._get_mt5_timeframe_from_string(timeframe_str)
    
    print(f"--- Executando ciclo de decisão para: {data_ticker} ({timeframe_str}) ---")

    # 1. Buscar dados recentes (usando o data_ticker)
    if data_ticker != order_ticker:
        print(f"Aviso: O ticker de dados ({data_ticker}) difere do ticker de ordens ({order_ticker}). Usando {data_ticker} para análise.")
    
    #latest_data = trader_instance.provider.get_latest_rates(data_ticker, 300, mt5_timeframe)
    
    if not mt5.initialize():
        print("Falha ao conectar ao MetaTrader 5")
        return

    # Se o horário de execução for após 18:30, usar os dados do dia anterior(114), se não, utilizar 0
    candle_position = 114 if datetime.now().time() > datetime.strptime("18:30", "%H:%M").time() else 0

    rates = mt5.copy_rates_from_pos(data_ticker, mt5_timeframe, candle_position, 300)
    latest_data = pd.DataFrame(rates)
    latest_data['time'] = pd.to_datetime(latest_data['time'], unit='s')
    latest_data.set_index('time', inplace=True)
    latest_data.rename(columns={'tick_volume': 'volume'}, inplace=True)

    if latest_data.empty: 
        print("Ticker de dados vazio, efetuando busca com o ticker de ordem.")
        latest_data = trader_instance.provider.get_latest_rates(trader_instance, order_ticker, 300, mt5_timeframe)
        if latest_data.empty:
            print("Ticker de ordem também vazio.")
            return


    print(f"Último candle recebido: {latest_data.index[-1]}")
    display(latest_data.tail(3))

    # 2. Gerar features
    featured_data = asset_state["strategy"].define_features(latest_data)
    X_live = featured_data[asset_state["strategy"].get_feature_names()].dropna()
    if X_live.empty: 
        print("Dados insuficientes para gerar features.")
        return

    # 3. Gerar sinal
    print("\nGerando sinal com o modelo de IA...")
    signal = asset_state["model"].predict(X_live)[-1]
    signal_text = 'COMPRA' if signal == 1 else 'VENDA'
    print(f"==> SINAL GERADO: {signal_text} ({signal}) ==<")

    # 4. Lógica de decisão
    if asset_state["position"] is None:
        if signal == 1: trader_instance._execute_trade(data_ticker, 'BUY')
        elif signal == 0: trader_instance._execute_trade(data_ticker, 'SELL')
    else:
        print(f"Posição já aberta para {order_ticker} ({asset_state['position']}).")
        
    print("--- Ciclo de decisão concluído ---")

### Passo 3: Executar Único Ciclo de Decisão (Single Tick)

In [19]:
print("\n--- Iniciando Teste de Ciclo Único ---")
try:
    if is_initialized and asset_state_para_testar:
        run_single_tick(trader, asset_state_para_testar)
        print("Teste concluído.")
        
except Exception as e:
    print(f"Erro durante o teste: {e}")


--- Iniciando Teste de Ciclo Único ---
--- Executando ciclo de decisão para: WDO$ (M5) ---
Aviso: O ticker de dados (WDO$) difere do ticker de ordens (WDOX25). Usando WDO$ para análise.
Último candle recebido: 2025-10-09 18:25:00


,open,high,low,close,volume,spread,real_volume
time,,,,,,,
2025-10-09 18:15:00,5405.5,5406.5,5402.5,5403.5,1814,1,7581
2025-10-09 18:20:00,5403.5,5407.5,5403.5,5404.0,1397,1,7555
2025-10-09 18:25:00,5404.5,5409.5,5403.5,5406.5,1383,1,7880



Gerando sinal com o modelo de IA...
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
==> SINAL GERADO: COMPRA (1) ==<
Erro durante o teste: 'dict' object has no attribute 'ticker_order'


### Passo 4: Encerrar a Conexão

In [13]:
print("Encerrando conexão com o MetaTrader 5...")
mt5.shutdown()
print("Conexão encerrada.")

Encerrando conexão com o MetaTrader 5...
Conexão encerrada.
